# BMW Repair Order — DSPy + MIPROv2 Prompt Optimization

**Approach:** DSPy signatures define the extraction task; MIPROv2 performs Bayesian search over candidate instructions and few-shot demonstrations, scored by `eval.py`.

**Contrast with try1.ipynb (GEPA):** GEPA reflects on individual failures and maintains a Pareto frontier; MIPROv2 optimizes over a labeled training batch using Bayesian optimization. Both use the same OCR cache, eval scorer, and 6 BMW repair order documents.

**Sections:**
1. Setup
2. OCR Intake (reuse cache from try1)
3. DSPy Signature and Module
4. Dataset and Metric
5. Baseline Evaluation
6. MIPROv2 Optimization
7. Inspect Optimized Instructions and Demos
8. Final Evaluation — Baseline vs Optimized
9. Visualization
10. Cross-Method Comparison (DSPy vs GEPA)
11. Artifact Export

## 1. Setup

In [ ]:
# Run once if packages are missing
!pip install -q dspy-ai deepdiff pdf2image surya-ocr

In [ ]:
import os, sys, json, re, pickle, hashlib, time, difflib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import dspy

# ── Paths ──────────────────────────────────────────────────────────────────────

def _find_repo_root(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent, start.parent.parent.parent]:
        if (p / "Data" / "Samples").exists() and (p / "Scripts" / "eval.py").exists():
            return p
    return start

BASE_DIR    = _find_repo_root(Path().resolve())
DATA_DIR    = BASE_DIR / "Data" / "Samples"
SCHEMA_PATH = BASE_DIR / "Data" / "extraction_schema.json"
SCRIPTS_DIR = BASE_DIR / "Scripts"
OCR_CACHE   = BASE_DIR / "Experimentation" / "Ivy" / "ocr_cache.pkl"

print(f"BASE_DIR   : {BASE_DIR}")
print(f"eval.py    : {(SCRIPTS_DIR / 'eval.py').exists()}")
print(f"Schema     : {SCHEMA_PATH.exists()}")

# ── Eval import ────────────────────────────────────────────────────────────────
sys.path.insert(0, str(SCRIPTS_DIR))
from eval import evaluate_extraction

# ── Schema ─────────────────────────────────────────────────────────────────────
with open(SCHEMA_PATH) as f:
    SCHEMA_STR = json.dumps(json.load(f), indent=2)
print(f"Schema: {len(SCHEMA_STR):,} chars")

# ── API key + DSPy LM ──────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

lm = dspy.LM("gemini/gemini-2.0-flash", api_key=GEMINI_API_KEY, temperature=0.0)
dspy.configure(lm=lm)
print(f"DSPy LM: {dspy.settings.lm}")

## 2. OCR Intake (Reuse Cache from try1.ipynb)

Uses the **new surya v0.17 API** (`FoundationPredictor` / `RecognitionPredictor` / `DetectionPredictor`). If `ocr_cache.pkl` was populated by try1.ipynb, all 6 docs will be cache hits — zero OCR calls.

In [ ]:
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor

print("Loading Surya OCR models (may take ~30 s on first run)...")
_foundation  = FoundationPredictor()
_recognition = RecognitionPredictor(_foundation)
_detection   = DetectionPredictor()
print("Models loaded.")


def pdf_to_text(pdf_path: Path) -> str:
    from pdf2image import convert_from_path
    pages  = convert_from_path(str(pdf_path))
    images = [p.convert("RGB") for p in pages]
    results = _recognition(images, det_predictor=_detection)
    page_texts = [
        "\n".join(line.text for line in result.text_lines)
        for result in results
    ]
    return "\n\n--- PAGE BREAK ---\n\n".join(page_texts)


# ── Cache load ─────────────────────────────────────────────────────────────────
OCR_CACHE.parent.mkdir(parents=True, exist_ok=True)
ocr_cache: dict = {}
if OCR_CACHE.exists():
    with open(OCR_CACHE, "rb") as f:
        ocr_cache = pickle.load(f)
    print(f"Loaded OCR cache: {len(ocr_cache)} entries.")

# ── Auto-discover PDF/JSON pairs and OCR ───────────────────────────────────────
PDF_FILES: dict[str, Path] = {}
GT_FILES:  dict[str, dict] = {}

for pdf_path in sorted(DATA_DIR.glob("*.pdf")):
    doc_id    = pdf_path.stem
    json_path = DATA_DIR / f"{doc_id}.json"
    if json_path.exists():
        PDF_FILES[doc_id] = pdf_path
        with open(json_path) as f:
            GT_FILES[doc_id] = json.load(f)

ocr_texts: dict[str, str] = {}
for doc_id, pdf_path in sorted(PDF_FILES.items()):
    md5 = hashlib.md5(pdf_path.read_bytes()).hexdigest()
    if md5 in ocr_cache:
        print(f"  {doc_id}: cache hit")
        ocr_texts[doc_id] = ocr_cache[md5]
    else:
        print(f"  {doc_id}: running OCR...", end=" ", flush=True)
        text = pdf_to_text(pdf_path)
        ocr_texts[doc_id] = text
        ocr_cache[md5]    = text
        with open(OCR_CACHE, "wb") as f:
            pickle.dump(ocr_cache, f)
        print(f"done ({len(text):,} chars)")

print(f"\n{len(ocr_texts)} docs ready: {sorted(ocr_texts.keys())}")

## 3. DSPy Signature and Extraction Module

**Design choices:**
- Schema passed as an `InputField` (`schema_hint`) rather than embedded in the docstring — MIPROv2 will optimize the docstring without perturbing the 644-line schema
- `ChainOfThought` (not `Predict`) gives MIPROv2 a richer search space: it can optimize both the reasoning instruction and the output instruction
- Output typed as `str` (not `dict`) — JSON parsing happens in the metric, keeping DSPy's internals clean

In [ ]:
class RepairOrderExtraction(dspy.Signature):
    """Extract all fields from a BMW repair order OCR text into a single complete
    JSON object. Capture every section (ASI/BWO/CSI/JSI/WSI/ISI), each job line,
    all operations, parts, labor entries, and log entries. Numeric fields must be
    numbers (int/float), not strings. Use null for absent optional fields.
    Output only the JSON — no markdown fences, no explanation."""

    ocr_text: str = dspy.InputField(
        desc="OCR text from the BMW repair order PDF. Pages separated by '--- PAGE BREAK ---'."
    )
    schema_hint: str = dspy.InputField(
        desc="JSON schema the output must conform to exactly",
        prefix="JSON Schema (conform exactly):\n",
    )
    extracted_json: str = dspy.OutputField(
        desc="Complete extracted JSON string, parseable by json.loads(). Must start with { and end with }."
    )


class RepairOrderExtractor(dspy.Module):
    def __init__(self):
        super().__init__()
        self.extract = dspy.ChainOfThought(RepairOrderExtraction)

    def forward(self, ocr_text: str, schema_hint: str) -> dspy.Prediction:
        return self.extract(ocr_text=ocr_text, schema_hint=schema_hint)


extractor = RepairOrderExtractor()
print("RepairOrderExtractor module defined.")

## 4. Dataset Construction and Metric Function

**Train/val split (4/2, fixed — not randomized):**
- Train: `201414, 344098, 629903, 678856`
- Val (holdout): `809570, 944962`

Fixed split ensures reproducible comparison against try1.ipynb (GEPA). `ground_truth` and `doc_id` are labels — excluded from `.with_inputs(...)` so they are never forwarded to the LM.

In [ ]:
TRAIN_IDS = ["201414", "344098", "629903", "678856"]
VAL_IDS   = ["809570", "944962"]


def make_example(doc_id: str) -> dspy.Example:
    return dspy.Example(
        ocr_text=ocr_texts[doc_id],
        schema_hint=SCHEMA_STR,
        ground_truth=GT_FILES[doc_id],   # label — not passed to LM
        doc_id=doc_id,                   # for logging
    ).with_inputs("ocr_text", "schema_hint")


trainset = [make_example(d) for d in TRAIN_IDS]
valset   = [make_example(d) for d in VAL_IDS]
allset   = trainset + valset

print(f"Train ({len(trainset)}): {TRAIN_IDS}")
print(f"Val   ({len(valset)}):   {VAL_IDS}")


def dspy_metric(example: dspy.Example, pred: dspy.Prediction, trace=None) -> float:
    """Bridge between DSPy prediction and eval.py scorer. Returns score in [0, 1]."""
    raw = getattr(pred, "extracted_json", "") or ""
    raw = raw.strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)

    try:
        pred_dict = json.loads(raw)
    except (json.JSONDecodeError, ValueError):
        return 0.0

    try:
        result = evaluate_extraction(example.ground_truth, pred_dict)
        return min(1.0, max(0.0, float(result.get("score", 0.0))))
    except Exception as exc:
        print(f"    [metric] eval.py exception: {exc}")
        return 0.0


print("Dataset and metric defined.")

In [ ]:
# ── Smoke test (1 API call) ────────────────────────────────────────────────────
print("Running smoke test on 1 example...")
smoke_ex   = trainset[0]
smoke_pred = extractor(ocr_text=smoke_ex.ocr_text, schema_hint=smoke_ex.schema_hint)
smoke_score = dspy_metric(smoke_ex, smoke_pred)

print(f"Smoke-test doc={smoke_ex.doc_id}  score={smoke_score:.3f}")
print(f"Raw output (first 400 chars): {smoke_pred.extracted_json[:400]}")

if smoke_score == 0.0:
    print("\nWARNING: score=0.0 — check if JSON is parseable:")
    try:
        raw = re.sub(r"^```(?:json)?\s*", "", smoke_pred.extracted_json.strip())
        raw = re.sub(r"\s*```$", "", raw)
        json.loads(raw)
        print("  JSON is valid — eval.py may have thrown an exception.")
    except json.JSONDecodeError as e:
        print(f"  JSON parse error: {e}")

## 5. Baseline Evaluation (Unoptimized Module)

In [ ]:
def evaluate_all_docs(program, label: str) -> dict:
    """Run program on all 6 docs. Returns {doc_id: {score, subscores, issues}}."""
    results = {}
    for doc_id in sorted(PDF_FILES.keys()):
        pred = program(ocr_text=ocr_texts[doc_id], schema_hint=SCHEMA_STR)
        raw  = getattr(pred, "extracted_json", "").strip()
        raw  = re.sub(r"^```(?:json)?\s*", "", raw)
        raw  = re.sub(r"\s*```$", "", raw)
        try:
            pred_dict = json.loads(raw)
            report    = evaluate_extraction(GT_FILES[doc_id], pred_dict)
        except Exception as exc:
            report = {"score": 0.0, "subscores": {}, "issues": [{"detail": str(exc)}]}
        results[doc_id] = report
        split = "TRAIN" if doc_id in TRAIN_IDS else "VAL"
        print(f"  [{label}] {doc_id} ({split}): score={report['score']:.3f}")

    mean = np.mean([r["score"] for r in results.values()])
    val_mean = np.mean([results[d]["score"] for d in VAL_IDS if d in results])
    print(f"  Mean all-6={mean:.3f}  Val-only={val_mean:.3f}")
    return results


print("Running baseline (6 API calls)...")
baseline_results = evaluate_all_docs(extractor, "baseline")

In [ ]:
# ── Baseline subscore table ────────────────────────────────────────────────────
DOC_IDS = sorted(PDF_FILES.keys())
cats    = ["structure", "numbers", "text"]

print(f"\n{'Doc':>10} | {'Score':>7} | {'Struct':>8} | {'Numbers':>8} | {'Text':>7} | Split")
print("-" * 60)
for doc_id in DOC_IDS:
    r  = baseline_results[doc_id]
    sub = r.get("subscores", {})
    split = "TRAIN" if doc_id in TRAIN_IDS else "VAL"
    print(f"{doc_id:>10} | {r['score']:>7.3f} | "
          f"{sub.get('structure',0):>8.3f} | {sub.get('numbers',0):>8.3f} | "
          f"{sub.get('text',0):>7.3f} | {split}")

## 6. MIPROv2 Optimization

**Parameter choices** (conservative for 4-doc training set):

| Parameter | Value | Rationale |
|---|---|---|
| `num_candidates` | 6 | 6 instruction candidates × 4 train = 24 scoring calls |
| `max_bootstrapped_demos` | 1 | With 4 train docs, >1 demo would overfit |
| `max_labeled_demos` | 1 | Same |
| `num_trials` | 12 | 12 Bayesian trials × 4 train = 48 calls; coarser but tractable |
| `seed` | 42 | Reproducibility for GEPA comparison |

**Total API calls:** ~76 optimization + 6 baseline + 6 final eval ≈ **88 extraction calls**

In [ ]:
NUM_CANDIDATES    = 6
MAX_BOOT_DEMOS    = 1
MAX_LABELED_DEMOS = 1
NUM_TRIALS        = 12

est_opt = NUM_CANDIDATES * len(trainset) + NUM_TRIALS * len(trainset) + MAX_BOOT_DEMOS * len(trainset)
print(f"Estimated optimization calls: ~{est_opt} extraction + ~{NUM_CANDIDATES} instruction-gen")
print(f"Total budget (incl. baseline + final eval): ~{est_opt + 12}")

In [ ]:
from dspy.teleprompt import MIPROv2

teleprompter = MIPROv2(
    metric=dspy_metric,
    num_candidates=NUM_CANDIDATES,
    max_bootstrapped_demos=MAX_BOOT_DEMOS,
    max_labeled_demos=MAX_LABELED_DEMOS,
    num_trials=NUM_TRIALS,
    seed=42,
    verbose=True,
)

print("Starting MIPROv2 optimization...")
print(f"Train: {TRAIN_IDS}  |  Val: {VAL_IDS}")

try:
    optimized_extractor = teleprompter.compile(
        extractor,
        trainset=trainset,
        valset=valset,
        requires_permission_to_run=False,  # suppress TTY prompt in Colab
    )
except TypeError:
    # older DSPy versions don't have requires_permission_to_run
    optimized_extractor = teleprompter.compile(
        extractor,
        trainset=trainset,
        valset=valset,
    )

print("\nMIPROv2 optimization complete.")

# Save compiled program
SAVE_PATH = BASE_DIR / "Experimentation" / "Ivy" / "try2_optimized.json"
optimized_extractor.save(str(SAVE_PATH))
print(f"Optimized program saved to: {SAVE_PATH}")

## 7. Inspect Optimized Instructions and Demos

In [ ]:
opt_signature = optimized_extractor.extract.signature

print("=" * 70)
print("OPTIMIZED INSTRUCTION:")
print("=" * 70)
print(opt_signature.instructions)

print("\n" + "=" * 70)
print("SELECTED DEMOS (few-shot examples chosen by MIPROv2):")
print("=" * 70)
demos = getattr(optimized_extractor.extract, "demos", [])
if demos:
    for i, demo in enumerate(demos):
        print(f"\n--- Demo {i+1} ---")
        if hasattr(demo, "__dict__"):
            demo_dict = vars(demo)
        else:
            demo_dict = demo
        ocr_snippet = str(demo_dict.get("ocr_text", ""))[:200]
        out_snippet = str(demo_dict.get("extracted_json", ""))[:300]
        print(f"  Input OCR (first 200 chars): {ocr_snippet}")
        print(f"  Output JSON (first 300 chars): {out_snippet}")
else:
    print("  (No demos selected — MIPROv2 achieved improvement via instruction-only optimization)")

In [ ]:
# ── Diff: original instruction vs optimized ────────────────────────────────────
orig_instr = RepairOrderExtraction.__doc__.strip()
opt_instr  = opt_signature.instructions.strip()

diff = list(difflib.unified_diff(
    orig_instr.splitlines(keepends=True),
    opt_instr.splitlines(keepends=True),
    fromfile="Original Instruction (Signature docstring)",
    tofile="Optimized Instruction (MIPROv2 output)",
    lineterm="",
))

if diff:
    print("Instruction diff:")
    for line in diff[:80]:
        print(line)
    if len(diff) > 80:
        print(f"... ({len(diff)-80} more lines)")
else:
    print("No change detected — MIPROv2 kept the original instruction.")
    print("Consider increasing num_candidates or num_trials, or using temperature=0.1 on the LM.")

## 8. Final Evaluation — Baseline vs Optimized

In [ ]:
print("Running optimized extractor on all 6 docs (6 API calls)...")
optimized_results = evaluate_all_docs(optimized_extractor, "optimized")

In [ ]:
# ── Score comparison table ─────────────────────────────────────────────────────
b_scores = [baseline_results[d]["score"]  for d in DOC_IDS]
o_scores = [optimized_results[d]["score"] for d in DOC_IDS]
b_mean   = np.mean(b_scores)
o_mean   = np.mean(o_scores)
b_val    = np.mean([baseline_results[d]["score"]  for d in VAL_IDS])
o_val    = np.mean([optimized_results[d]["score"] for d in VAL_IDS])

print(f"\n{'Doc':>10} | {'Baseline':>10} | {'Optimized':>10} | {'Delta':>8} | Split")
print("-" * 56)
for doc_id in DOC_IDS:
    b = baseline_results[doc_id]["score"]
    o = optimized_results[doc_id]["score"]
    split = "TRAIN" if doc_id in TRAIN_IDS else "VAL"
    print(f"{doc_id:>10} | {b:>10.3f} | {o:>10.3f} | {o-b:>+8.3f} | {split}")
print("-" * 56)
print(f"{'All-6 mean':>10} | {b_mean:>10.3f} | {o_mean:>10.3f} | {o_mean-b_mean:>+8.3f} |")
print(f"{'Val mean':>10} | {b_val:>10.3f} | {o_val:>10.3f} | {o_val-b_val:>+8.3f} | ← headline"
      " (generalization)")

# ── Subscore table ─────────────────────────────────────────────────────────────
print(f"\n{'Category':>12} | {'Baseline':>10} | {'Optimized':>10} | {'Delta':>8}")
print("-" * 48)
cat_weights = {"structure": 0.45, "numbers": 0.40, "text": 0.15}
for cat, w in cat_weights.items():
    b_cat = np.mean([baseline_results[d].get("subscores", {}).get(cat, 0)  for d in DOC_IDS])
    o_cat = np.mean([optimized_results[d].get("subscores", {}).get(cat, 0) for d in DOC_IDS])
    print(f"{cat+f' (w={w})':>12} | {b_cat:>10.3f} | {o_cat:>10.3f} | {o_cat-b_cat:>+8.3f}")

## 9. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Panel 1: grouped bar chart ─────────────────────────────────────────────────
x     = np.arange(len(DOC_IDS))
width = 0.35

ax1 = axes[0]
bars_b = ax1.bar(x - width/2, b_scores, width, label="Baseline",  color="steelblue", alpha=0.85)
bars_o = ax1.bar(x + width/2, o_scores, width, label="Optimized", color="seagreen",  alpha=0.85)
ax1.axhline(y=b_mean, color="steelblue", linestyle="--", alpha=0.5, linewidth=1.2,
            label=f"Baseline mean={b_mean:.3f}")
ax1.axhline(y=o_mean, color="seagreen",  linestyle="--", alpha=0.5, linewidth=1.2,
            label=f"Optimized mean={o_mean:.3f}")
ax1.set_xticks(x)
ax1.set_xticklabels(
    [f"{d}\n({'T' if d in TRAIN_IDS else 'V'})" for d in DOC_IDS], fontsize=8
)
ax1.set_ylabel("Score (0–1)")
ax1.set_title("Baseline vs MIPROv2-Optimized\nPer-Doc Score  (T=train, V=val)")
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.25, axis="y")

# ── Panel 2: subscore heatmap ──────────────────────────────────────────────────
# Rows = category, cols = baseline docs then optimized docs
categories = list(cat_weights.keys())
col_labels  = [f"{d}\nBase" for d in DOC_IDS] + [f"{d}\nOpt" for d in DOC_IDS]
matrix = np.array([
    [baseline_results[d].get("subscores", {}).get(cat, 0)  for d in DOC_IDS]
    + [optimized_results[d].get("subscores", {}).get(cat, 0) for d in DOC_IDS]
    for cat in categories
])

ax2 = axes[1]
im  = ax2.imshow(matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax2.set_xticks(range(len(col_labels)))
ax2.set_xticklabels(col_labels, fontsize=7, rotation=45, ha="right")
ax2.set_yticks(range(len(categories)))
ax2.set_yticklabels([f"{c} (w={cat_weights[c]})" for c in categories], fontsize=8)
ax2.set_title("Subscore Heatmap (structure / numbers / text)\n← Baseline | Optimized →")
ax2.axvline(x=len(DOC_IDS) - 0.5, color="black", linewidth=1.5, alpha=0.6)
plt.colorbar(im, ax=ax2)

for i in range(len(categories)):
    for j in range(len(col_labels)):
        v = matrix[i, j]
        ax2.text(j, i, f"{v:.2f}", ha="center", va="center",
                 fontsize=7, color="black" if 0.25 < v < 0.85 else "white")

plt.tight_layout()
plot_path = BASE_DIR / "Experimentation" / "Ivy" / "try2_results.png"
plt.savefig(str(plot_path), dpi=150)
plt.show()
print(f"Saved to {plot_path}")

## 10. Cross-Method Comparison — DSPy+MIPROv2 vs GEPA

Loads the best Pareto frontier candidate from `gepa_artifact.json` (produced by try1.ipynb) and compares per-doc scores. A GEPA > DSPy result is consistent with the GEPA paper's reported +10% advantage and is a positive finding for the project.

In [ ]:
gepa_artifact_path = BASE_DIR / "Experimentation" / "Ivy" / "gepa_artifact.json"

if gepa_artifact_path.exists():
    with open(gepa_artifact_path) as f:
        gepa_artifact = json.load(f)

    best_gepa   = max(gepa_artifact["pareto_frontier"], key=lambda x: x["mean_score"])
    gepa_by_doc = best_gepa["per_doc_scores"]  # {doc_id: score}
    g_mean = np.mean(list(gepa_by_doc.values()))

    print(f"GEPA best: [{best_gepa['prompt_id']}]  iter={best_gepa['iteration']}  mean={best_gepa['mean_score']:.3f}")
    print(f"DSPy base mean={b_mean:.3f}   DSPy opt mean={o_mean:.3f}\n")

    print(f"{'Doc':>10} | {'GEPA':>8} | {'DSPy-base':>10} | {'DSPy-opt':>10} | {'GEPA-DSPy':>10}")
    print("-" * 60)
    for doc_id in DOC_IDS:
        g = gepa_by_doc.get(doc_id, float("nan"))
        b = baseline_results[doc_id]["score"]
        o = optimized_results[doc_id]["score"]
        print(f"{doc_id:>10} | {g:>8.3f} | {b:>10.3f} | {o:>10.3f} | {g-o:>+10.3f}")
    print("-" * 60)
    print(f"{'Mean':>10} | {g_mean:>8.3f} | {b_mean:>10.3f} | {o_mean:>10.3f} | {g_mean-o_mean:>+10.3f}")

    print("\nAPI call efficiency comparison:")
    est_gepa_calls = 6 + 5 * 7   # baseline + 5 iters × (1 reflection + 6 extraction)
    est_dspy_calls = 1 + 6 + (NUM_CANDIDATES * len(trainset) + NUM_TRIALS * len(trainset)) + 6
    print(f"  GEPA:          ~{est_gepa_calls} extraction calls + 5 reflection calls")
    print(f"  DSPy/MIPROv2:  ~{est_dspy_calls} extraction calls + ~{NUM_CANDIDATES} instruction-gen calls")
else:
    print("gepa_artifact.json not found.")
    print("Run try1.ipynb (Sections 1–9) first to generate it, then re-run this cell.")

## 11. Artifact Export

In [ ]:
artifact = {
    "metadata": {
        "method":    "DSPy+MIPROv2",
        "model":     "gemini-2.0-flash",
        "train_ids": TRAIN_IDS,
        "val_ids":   VAL_IDS,
        "miprov2_params": {
            "num_candidates":       NUM_CANDIDATES,
            "max_bootstrapped_demos": MAX_BOOT_DEMOS,
            "max_labeled_demos":    MAX_LABELED_DEMOS,
            "num_trials":           NUM_TRIALS,
            "seed":                 42,
        },
    },
    "baseline": {
        doc_id: {
            "score":    baseline_results[doc_id]["score"],
            "subscores": baseline_results[doc_id].get("subscores", {}),
        }
        for doc_id in DOC_IDS
    },
    "optimized": {
        doc_id: {
            "score":    optimized_results[doc_id]["score"],
            "subscores": optimized_results[doc_id].get("subscores", {}),
        }
        for doc_id in DOC_IDS
    },
    "optimized_instruction": opt_signature.instructions,
    "n_demos_selected":      len(getattr(optimized_extractor.extract, "demos", [])),
    "summary": {
        "baseline_mean":  round(b_mean, 4),
        "optimized_mean": round(o_mean, 4),
        "delta_mean":     round(o_mean - b_mean, 4),
        "val_baseline":   round(b_val, 4),
        "val_optimized":  round(o_val, 4),
        "val_delta":      round(o_val - b_val, 4),
    },
}

artifact_path = BASE_DIR / "Experimentation" / "Ivy" / "try2_artifact.json"
with open(artifact_path, "w") as f:
    json.dump(artifact, f, indent=2, default=str)
print(f"Artifact saved: {artifact_path}")
print(f"Summary: baseline_mean={b_mean:.3f}  optimized_mean={o_mean:.3f}  "
      f"delta={o_mean-b_mean:+.3f}  val_delta={o_val-b_val:+.3f}")